# 01 — Transformer from scratch: attention, GPT, RoPE, GQA

Interactive version of the repo's demo scripts. Run cells top to bottom;
everything runs on CPU or Apple MPS automatically (no CUDA required).

Repo: [`src/transformer/model.py`](../src/transformer/model.py), [`src/transformer/gpt.py`](../src/transformer/gpt.py),
[`scripts/verify.py`](../scripts/verify.py), [`scripts/train_seq2seq.py`](../scripts/train_seq2seq.py), [`README.md`](../README.md)

## 0. Setup

No HuggingFace, no `nn.Transformer` — the math is written out, so you can read every line of it.

In [ ]:
import math
import os
import sys
import time
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Repo root = parent of the notebook directory (kernel cwd = notebooks/)
ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, 'src'))
CKPT = os.path.join(ROOT, 'checkpoints')

from transformer import (Transformer, create_masks, create_look_ahead_mask,
                         precompute_rope, apply_rope)
from transformer.gpt import GPT, CharTokenizer
                              # CharTokenizer: checkpoints pickle it as
                              # __main__.CharTokenizer — torch.load needs it
                              # resolvable in this namespace

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## 1. The primitive: scaled dot-product attention

Attention output for a query = weighted average of values; weights come
from the dot product of query with keys, softened by softmax. The
`/sqrt(d_k)` keeps the softmax from saturating to one-hot (see
`src/transformer/attention.py`: max-P 0.47 → 0.98 as d_k grows
8 → 4096 without the scale; flat 0.125 with it).

In [ ]:
torch.manual_seed(0)
B, H, T, d_k = 1, 1, 4, 8
Q = torch.randn(B, H, T, d_k); K = torch.randn(B, H, T, d_k); V = torch.randn(B, H, T, d_k)
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
P = F.softmax(scores, dim=-1)
print('attention weights (rows=queries, cols=keys):')
print(torch.round(P[0, 0], decimals=2))

mask = create_look_ahead_mask(T)  # causal: token i attends to keys <= i
P_masked = F.softmax(scores.masked_fill(~mask, float('-inf')), dim=-1)
print('causal version:')
print(torch.round(P_masked[0, 0], decimals=2))
assert torch.allclose(P_masked[0, 0].tril(), P_masked[0, 0], atol=1e-6)
print('lower-triangular: OK')

## 2. The seq2seq Transformer

`src/transformer/model.py` builds the full encoder–decoder from the
2017 paper. `scripts/train_seq2seq.py` teaches it reverse/copy on small
integer sequences. We load the trained `checkpoints/model.pt` and look
at what attention actually learned.

In [ ]:
model = Transformer(src_vocab_size=20, tgt_vocab_size=20, d_model=64,
                    num_heads=4, d_ff=128, num_layers=2, max_len=6)
model.load_state_dict(torch.load(os.path.join(CKPT, 'model.pt'), map_location='cpu'))
model.eval()
print(f'params: {model.count_params():,}')

In [ ]:
src = torch.tensor([[9, 15, 3, 8, 5]])
tgt_input = torch.tensor([[1, 5, 8, 3, 15]])  # <SOS> + target[:-1]
src_mask, tgt_mask = create_masks(src, tgt_input, pad_idx=0)
with torch.no_grad():
    enc_out = model.encoder(src, src_mask)
    model.decoder(tgt_input, enc_out, tgt_mask, src_mask)

w = model.decoder.layers[0].cross_attn.last_attn_weights[0][0]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(w, cmap='viridis')
ax.set_yticks(range(5)); ax.set_yticklabels([f'q={t}' for t in tgt_input[0].tolist()])
ax.set_xticks(range(5)); ax.set_xticklabels([f'k={t}' for t in src[0].tolist()])
ax.set_title('Decoder cross-attention, layer 0 (head 0)')
plt.colorbar(im, ax=ax)
plt.show()
# A reverse model attends to the source token it needs for THIS output slot.

## 3. Decoder-only GPT: Shakespeare in ~800k params

`src/transformer/gpt.py` keeps only the masked self-attention half and
trains a char-level LM on `data/shakespeare.txt`. We load the RoPE
checkpoint `checkpoints/gpt_rope.pt`.

In [ ]:
ckpt = torch.load(os.path.join(CKPT, 'gpt_rope.pt'), map_location='cpu', weights_only=False)
tokenizer = ckpt['tokenizer']  # unpickled as __main__.CharTokenizer
gpt = GPT(vocab_size=tokenizer.vocab_size, max_len=128, rope=True,
          num_kv_heads=ckpt.get('num_kv_heads'))
gpt.load_state_dict(ckpt['model'])
gpt.to(device).eval()
print(f'vocab: {tokenizer.vocab_size} chars | params: {gpt.count_params():,}')
print(f'val loss: {ckpt["val_loss"]:.4f}')

In [ ]:
prompt = 'To be, or not to be'
idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
out = gpt.generate(idx, 120, temperature=0.9)
print(tokenizer.decode(out[0].tolist()))

## 4. KV cache: correctness by construction

`generate_cached` prefill processes the whole prompt **with the causal
mask** (the original bug — no mask ⇒ wrong hidden states everywhere),
then one forward pass per new token with K/V appended to the cache.
The cache must be indistinguishable from recomputing attention:

- prefill: bit-identical (max |Δ| = 0.0)
- one-token step: |Δ| < 1e-4
- same seed ⇒ identical sampled text on both paths

In [ ]:
with torch.no_grad():
    mask = create_look_ahead_mask(idx.size(1)).to(device)
    naive = gpt(idx, mask)
    caches = [None] * len(gpt.blocks)
    x, caches = gpt._cached_forward(idx, caches, 0, mask)
    cached = gpt.lm_head(gpt.ln_f(x))
print('prefill max |\u0394|:', (naive - cached).abs().max().item())

with torch.no_grad():
    nxt = cached.argmax(-1)[:, -1:]
    x, caches = gpt._cached_forward(nxt, caches, idx.size(1))
    step = gpt.lm_head(gpt.ln_f(x))
    seq = torch.cat([idx, nxt], dim=1)
    step_naive = gpt(seq, create_look_ahead_mask(seq.size(1)).to(device))[:, -1:]
print('one-token step max |\u0394|:', (step - step_naive).abs().max().item())

torch.manual_seed(7); a = gpt.generate(idx, 60, temperature=0.9)
torch.manual_seed(7); b = gpt.generate_cached(idx, 60, temperature=0.9)
print('same-seed sampling identical:', torch.equal(a, b))

t0 = time.time(); gpt.generate(idx, 60, temperature=0.9); t1 = time.time()
gpt.generate_cached(idx, 60, temperature=0.9); t2 = time.time()
print(f'naive {t1-t0:.2f}s vs cached {t2-t1:.2f}s \u2192 {(t1-t0)/(t2-t1):.1f}x faster')

## 5. RoPE: positions as rotations

RoPE rotates Q and K instead of adding position vectors. The score
between tokens m and n then depends only on the offset m−n (Toeplitz),
which is why positions past the training window still work. Proof next.

In [ ]:
d_model, max_len = 32, 16
cos, sin = precompute_rope(d_model, max_len)
q = torch.randn(1, d_model); k = torch.randn(1, d_model)
scores = torch.zeros(max_len, max_len)
for m in range(max_len):
    q_m = apply_rope(q, cos[m:m+1], sin[m:m+1])
    for n in range(max_len):
        scores[m, n] = (q_m * apply_rope(k, cos[n:n+1], sin[n:n+1])).sum()
dev = (scores[1:, 1:] - scores[:-1, :-1]).abs().max().item()
print(f'score[m+1,n+1] == score[m,n]? max deviation {dev:.2e}')

In [ ]:
n_new = 170 - idx.size(1)  # 42 tokens past the 128-token training window
torch.manual_seed(7)
out = gpt.generate_cached(idx, n_new, temperature=0.9)
print(f'generated {out.size(1)} tokens ({out.size(1) - 128} past the window)')
print(tokenizer.decode(out[0].tolist())[:150], '...')

# learned-position models cannot:
gpt_plain = GPT(vocab_size=tokenizer.vocab_size, max_len=128, rope=False)
try:
    gpt_plain.generate_cached(idx[:, :5], 200, temperature=0.9)
except ValueError as e:
    print('learned positions:', e)

## 6. GQA: shrink the KV cache

With `--kv-heads 2` and 4 query heads, pairs of query heads share one
K/V head. The cache stores the SMALL form `(B, kv_heads, T, d_k)` and
expands per-head only for scoring. Cache: 512 KiB → 256 KiB at 128 ctx
on the trained checkpoints; quality: ppl 4.49 → 4.64.

In [ ]:
gqa_ckpt = torch.load(os.path.join(CKPT, 'gpt_rope_gqa.pt'), map_location='cpu', weights_only=False)
gqa = GPT(vocab_size=gqa_ckpt['tokenizer'].vocab_size, max_len=128, rope=True,
          num_kv_heads=gqa_ckpt.get('num_kv_heads'))
gqa.load_state_dict(gqa_ckpt['model'])
gqa.to(device).eval()
attn = gqa.blocks[0].self_attn
print(f'{gqa.num_heads} query heads \u2192 {attn.num_kv_heads} KV heads, '
      f'group_size={attn.group_size}, d_k={attn.d_k}')

with torch.no_grad():
    idx1 = torch.randint(0, gqa_ckpt['tokenizer'].vocab_size, (1, 12), device=device)
    mask1 = create_look_ahead_mask(12).to(device)
    naive1 = gqa(idx1, mask1)
    caches = [None] * len(gqa.blocks)
    x, caches = gqa._cached_forward(idx1, caches, 0, mask1)
    prefill = gqa.lm_head(gqa.ln_f(x))
    nxt = naive1[:, -1:].argmax(-1)
    x, caches = gqa._cached_forward(nxt, caches, 12)
    step = gqa.lm_head(gqa.ln_f(x))
    seq1 = torch.cat([idx1, nxt], dim=1)
    step_naive = gqa(seq1, create_look_ahead_mask(13).to(device))[:, -1:]
print('prefill max |\u0394|:', (naive1 - prefill).abs().max().item())
print('one-token step max |\u0394|:', (step - step_naive).abs().max().item())
print('cached K shape:', tuple(caches[0][0].shape), '— small form, '
      f'{attn.num_kv_heads}/{gqa.num_heads} of full-head size')

## 7. What's next

Run the full verification harness for the whole story:

```bash
python scripts/verify.py                            # learned positions
python scripts/verify.py --rope --ckpt checkpoints/gpt_rope.pt  # RoPE, full MHA
python scripts/verify.py --rope --ckpt checkpoints/gpt_rope_gqa.pt  # GQA
```

Each section is an equivalence proof (cached == naive, hand-rolled
GQA loop == vectorized path, same-seed sampling == same text).

Train your own:

```bash
python scripts/gpt.py --epochs 30 --rope --save --save-path my_model.pt --kv-heads 2
python scripts/gpt.py --load --rope --load-path my_model.pt --prompt 'To be' --top-p 0.9
```